# Workflow: a simple linear workflow

This is notebook 1 of 3 in the workflow series.

A workflow is a lightweight container that lists process steps and records
their order. This notebook walks through the full pipeline for a simple
three-step quality assurance workflow:

```
Step 1: Raw Material Inspection
           |
Step 2: Tensile Test
           |
Step 3: Quality Report
```

After working through this notebook, see:
- [Notebook 2](2_workflow_branching.ipynb): branching and parallel steps
- [Notebook 3](3_workflow_cross_domain.ipynb): a full cross-domain example


In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata

# Folder layout
HERE     = pathlib.Path().resolve()   # schemas/workflow/PMDCo/docs/
WORKFLOW = HERE.parent                # schemas/workflow/PMDCo/
SCHEMAS  = WORKFLOW.parents[1]        # schemas/

def load_context(schema_dir):
    return yaml.safe_load((schema_dir / 'specs' / 'schema.oold.yaml').read_text())['@context']

def transform(schema_dir, data):
    expr = (schema_dir / 'simplified' / 'transform.jsonata').read_text()
    return Jsonata(expr).evaluate(data)

def parse_to_graph(context, oold_doc):
    ds = rdflib.Dataset()
    ds.parse(data=json.dumps({'@context': context, **oold_doc}), format='json-ld')
    g = rdflib.Graph()
    for s, p, o, _ in ds.quads():
        g.add((s, p, o))
    return g

print('Schema folder:', WORKFLOW)
print('Exists:', WORKFLOW.exists())

## Step 1: Define the workflow

Fill in a workflow name and a list of steps. Each step needs at minimum a label.
Optional fields:

| Field | Purpose |
|---|---|
| `step_type` | ontology class CURIE for the step (e.g. `obi:0000070` for an assay) |
| `description` | free-text description |
| `conditions` | inline parameters (name, value, unit) |
| `instance_iri` | link to a detailed record in a knowledge graph (covered in notebook 3) |
| `preceded_by` | explicit predecessor step IDs for branching (covered in notebook 2) |


In [ ]:
workflow_input = {
    'workflow_name': 'QA workflow, batch A',
    'description': 'Three-step quality assurance workflow for a steel sample batch.',
    'steps': [
        {
            'label': 'Raw Material Inspection',
            'description': 'Visual and dimensional check of the incoming steel sheet.',
        },
        {
            'label': 'Tensile Test',
            'description': 'Room-temperature tensile test following ISO 6892-1.',
            'conditions': [
                {'name': 'Standard',    'unit': 'ISO 6892-1'},
                {'name': 'Temperature', 'value': 23, 'unit': 'degC'},
            ],
        },
        {
            'label': 'Quality Report',
            'description': 'Document test results and issue an accept or reject decision.',
        },
    ],
}

oold_doc = transform(WORKFLOW, workflow_input)

print('OO-LD document:')
print(json.dumps(oold_doc, indent=2))

The transform automatically links each step to the one before it
(`preceded_by` pointing to the previous step ID). The first step has no predecessor.

Notice that step IDs are derived from the workflow name and step labels.
You can override them with `step_id` (shown in notebook 2).


In [ ]:
print('Step order:')
for s in oold_doc['has_part']:
    pre = s.get('preceded_by', ['(first)'])
    print(f"  {s['id']}")
    print(f"    label       : {s['label']}")
    print(f"    preceded_by : {pre}")
    print()

## Step 2: Build the RDF graph

The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`. This maps JSON field names to ontology IRIs,
for example `has_part` becomes `bfo:BFO_0000051` and `preceded_by` becomes
`bfo:BFO_0000062`.


In [ ]:
context = load_context(WORKFLOW)
graph   = parse_to_graph(context, oold_doc)

print(f'Graph contains {len(graph)} triples.')
print()
print(graph.serialize(format='turtle'))

## Step 3: Validate with SHACL

The SHACL shape checks that:
- The workflow node has a label and at least one step
- Each step node has a label


In [ ]:
shapes = rdflib.Graph().parse(str(WORKFLOW / 'specs' / 'shape.ttl'))
conforms, report, _ = pyshacl.validate(graph, shacl_graph=shapes, inference='rdfs')

print(f'Conforms: {conforms}')
if not conforms:
    SH = rdflib.Namespace('http://www.w3.org/ns/shacl#')
    for res in report.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg = report.value(res, SH.resultMessage)
        print(f'  Violation: {msg}')

## Step 4: Query the graph

SPARQL is the query language for RDF graphs. The query below retrieves
all steps and their predecessors, showing the workflow order.


In [ ]:
SPARQL = """
PREFIX bfo:    <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?step_label ?preceded_by_label
WHERE {
  ?step rdfs:label ?step_label .
  FILTER NOT EXISTS { ?step dcterms:conformsTo ?_ . }  # exclude the workflow root
  OPTIONAL {
    ?step bfo:0000062 ?prev .
    ?prev rdfs:label ?preceded_by_label .
  }
}
ORDER BY ?step_label
"""

rows = list(graph.query(SPARQL))
print('{:<30}  {}'.format('Step', 'Preceded by'))
print('-' * 55)
for r in rows:
    pre = str(r.preceded_by_label) if r.preceded_by_label else '(first)'
    print(f'{str(r.step_label):<30}  {pre}')

## Summary

| Step | What happened |
|---|---|
| 1 | Defined a workflow with three steps as a plain Python dict |
| 2 | JSONata transform converted it to an OO-LD document |
| 3 | The OO-LD document was parsed into an RDF graph |
| 4 | SHACL validation confirmed the graph is structurally correct |
| 5 | A SPARQL query retrieved the step order from the graph |

---

**Next:** [Notebook 2](2_workflow_branching.ipynb) shows how to use `preceded_by`
to create branching and parallel workflows.
